# 🔬 τ-Knowledge Source Preparation (For Advanced Users / Operators)

This notebook is the first step of the **authoring path**.  
It prepares the sources needed to create the **prepared data bundle** used in learner notebooks (03, 04).

## Target Audience
- Operators / Advanced users / Data engineers
- Users who want to perform SDG (Synthetic Data Generation) themselves

## Processing Steps

1. Pin τ-bench version and inspect KB (knowledge base)
2. Extract policy facts
3. Inspect tool schemas
4. Review official task/split boundaries
5. Save source snapshot

### Key Principles
- **Private evaluation information** (expected actions, reward criteria, hidden user goals) must not be included in training data
- Official evaluation tasks are reserved for evaluation; independent training scenarios are generated
- Version, SHA, and license are recorded for all sources

In [ ]:
"""Pin τ-bench version and inspect KB."""

import os
import json
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_yaml_config, PROJECT_ROOT,
)

load_env()

# Load preparation config
prep_config = load_yaml_config("configs/data-preparation.yaml")
tau_config = prep_config["tau_bench"]

tau_version = tau_config.get("version", os.environ.get("TAU_BENCH_VERSION", ""))
tau_sha = tau_config.get("commit_sha", os.environ.get("TAU_BENCH_COMMIT_SHA", ""))
tau_domain = tau_config.get("domain", "banking_knowledge")

print("=" * 70)
print("📌 Pin τ-bench Version")
print("=" * 70)
print(f"  Version: {tau_version}")
print(f"  Commit SHA: {tau_sha or '(not set — pin after verification)'}")
print(f"  Domain: {tau_domain}")
print()

# Check τ-bench installation
tau_install_path = tau_config.get("install_path", os.environ.get("TAU_BENCH_INSTALL_PATH", ""))

if tau_install_path and Path(tau_install_path).exists():
    print(f"✅ τ-bench install path: {tau_install_path}")

    # Inspect KB
    kb_paths = list(Path(tau_install_path).rglob("*knowledge*"))
    print(f"\n   KB-related files:")
    for p in kb_paths[:10]:
        print(f"     {p.relative_to(tau_install_path)}")
else:
    print("⚠️  τ-bench install path is not set or does not exist.")
    print("  Set TAU_BENCH_INSTALL_PATH in your .env file.")
    print("  Or: pip install tau2-bench")
    print()
    print("  Reference: https://github.com/sierra-research/tau2-bench")

# Verify v1.0.1 grading correction
print("\n📋 Version Notes:")
print("  v1.0.1 includes a banking grading correction.")
print("  Scores before and after the correction are not directly comparable.")

In [ ]:
"""Extract policy facts from KB."""

source_config = prep_config["sources"]
kb_output = PROJECT_ROOT / source_config["kb_snapshot"]["output_path"]
policy_output = PROJECT_ROOT / source_config["policy_extraction"]["output_path"]

print("=" * 70)
print("📋 Policy Fact Extraction")
print("=" * 70)

kb_output.parent.mkdir(parents=True, exist_ok=True)
policy_output.parent.mkdir(parents=True, exist_ok=True)

if tau_install_path and Path(tau_install_path).exists():
    # Load KB documents
    print("Loading KB documents...")

    try:
        # Try loading from τ-bench API
        import sys
        sys.path.insert(0, tau_install_path)

        # Inspect KB structure
        print("\n  KB structure inspection:")
        print("    - Preserve document boundaries")
        print("    - Retain sections, links, and policy versions")
        print("    - Include conditions and exceptions")

        # Extract policy facts
        print("\n  Policy fact extraction:")
        print(f"    Review sample size: {source_config['policy_extraction']['review_sample_size']}")
        print(f"    Retain source offsets: {source_config['policy_extraction']['retain_source_offsets']}")
        print(f"    Retain document IDs: {source_config['policy_extraction']['retain_document_ids']}")

        # Note: actual extraction requires τ-bench APIs
        print("\n  ℹ️  Actual extraction is performed via τ-bench APIs.")
        print("     Manual execution:")
        print(f"     python scripts/prepare_tau_sources.py --config configs/data-preparation.yaml")

    except Exception as exc:
        print(f"\n  ⚠️  Failed to load KB: {exc}")
        print("     Please verify the τ-bench installation.")
else:
    print("⚠️  τ-bench not installed — cannot perform policy extraction.")
    print("   Manual execution: python scripts/prepare_tau_sources.py")

In [ ]:
"""Inspect tool schemas."""

print("=" * 70)
print("🔧 Tool Schema Inspection")
print("=" * 70)

if tau_install_path and Path(tau_install_path).exists():
    # Look for tool definitions
    tool_files = list(Path(tau_install_path).rglob("*tool*"))
    tool_files += list(Path(tau_install_path).rglob("*action*"))

    print(f"  Tool-related files: {len(tool_files)}")
    for tf in tool_files[:15]:
        print(f"    {tf.relative_to(tau_install_path)}")

    print("\n  Inspection items:")
    print("    ✓ Tool names and descriptions")
    print("    ✓ Parameter schemas (JSON Schema)")
    print("    ✓ Required/optional parameters")
    print("    ✓ Return value format")
    print("    ✓ State-changing tool identification")
    print("    ✓ Tool discovery behavior (preserve official behavior)")
else:
    print("⚠️  τ-bench not installed — cannot inspect tool schemas.")

print("\n  ⚠️  Official tool discovery behavior must be preserved.")
print("     If all tool schemas are pre-exposed for convenience,")
print("     it must be labeled as modified benchmark conditions.")

In [ ]:
"""Review official task/split boundaries."""

print("=" * 70)
print("📊 Review Official Task/Split Boundaries")
print("=" * 70)

split_config = prep_config["splits"]

print(f"  Split method: {split_config['method']}")
print(f"  Train ratio: {split_config['train_ratio']}")
print(f"  Validation ratio: {split_config['validation_ratio']}")
print(f"  Seed: {split_config['seed']}")
print(f"  Contamination check: {split_config['contamination_check']}")
print(f"  Family isolation: {split_config['family_isolation']}")

print("\n📋 Split Principles:")
print("  1. All official evaluation tasks are reserved for evaluation")
print("  2. If an official train split exists, verify the permitted conditions before use")
print("  3. Otherwise, generate training/validation data from KB and independent scenarios")
print("  4. Paraphrases, name/number substitutions, and sibling examples stay in the same split")
print("  5. Evaluation task wording, expected actions, and golden doc lists must never be used in SDG")

print("\n⚠️  Key Distinctions:")
print("  • kb_adaptation: Learn KB and apply to new situations → this experiment")
print("  • Task leakage: Using evaluation tasks/scenarios in training → forbidden")
print("  • KB sharing is intentional in kb_adaptation, distinct from task leakage")

In [ ]:
"""Save source snapshot."""

output_base = PROJECT_ROOT / prep_config["output"]["base_path"]
output_base.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("💾 Save Source Snapshot")
print("=" * 70)

# Save snapshot metadata
snapshot_meta = {
    "tau_version": tau_version,
    "tau_commit_sha": tau_sha,
    "domain": tau_domain,
    "split_config": split_config,
    "quality_gates": prep_config.get("quality_gates", {}),
    "output_paths": {
        "kb_snapshot": str(kb_output),
        "policy_extraction": str(policy_output),
    },
}

meta_path = output_base / "snapshot_metadata.json"
with open(meta_path, "w") as f:
    json.dump(snapshot_meta, f, indent=2, ensure_ascii=False)

print(f"✅ Snapshot metadata saved: {meta_path}")
print(f"   Output directory: {output_base}")

# Show quality gates
quality_gates = prep_config.get("quality_gates", {})
print("\n📊 Quality Gates:")
print(f"  Min acceptance rate: {quality_gates.get('min_acceptance_rate', 'N/A')}")
print(f"  Required types: {quality_gates.get('required_types', [])}")
print(f"  Min tool examples: {quality_gates.get('min_tool_examples', 'N/A')}")
print(f"  Min trajectory examples: {quality_gates.get('min_trajectory_examples', 'N/A')}")
print(f"  Human review sample size: {quality_gates.get('human_review_sample_size', 'N/A')}")

print("\nNext steps:")
print("  📓 02_generate_synthetic.ipynb — Synthetic Data Generation")
print("  Or CLI: python scripts/prepare_tau_sources.py --config configs/data-preparation.yaml")